# WildSpeech — Master Pipeline (local, GPU-accelerated)

Orchestrates stages **P00–P15** with dependency resolution, fingerprint-based caching and resume.

| Profile | Purpose |
|---|---|
| `SMOKE` | technical verification on a tiny deterministic subsample |
| `FAST` | full tabular analysis + inexpensive GPU acoustic families + baselines |
| `FULL` | + pitch, wavelets, robustness, LOSO, multimodal fusion |
| `RESEARCH` | + local ASR/alignment, foundation embeddings, neural MIL (opt-in, licence-gated) |
| `RESUME` | reload the last run configuration and continue |
| `SUBMISSION` | independent test inference with the frozen champion (P13 → P14) |

Expensive work (phonetics, foundation models, neural training, test inference) is **off by default**.
All data stays on this machine. Run cells top to bottom; nothing executes on import.

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "configs" / "default.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "configs" / "default.yaml").exists(), "open this notebook from inside the WildSpeech_GPU repository"
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("PYTHONUTF8", "1")
import pandas as pd
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
print("repository:", ROOT)

In [ ]:
# ---- run configuration -------------------------------------------------------
PROFILE = "SMOKE"          # SMOKE | FAST | FULL | RESEARCH | RESUME | SUBMISSION
STAGES = None              # None = the profile's default stages, e.g. ["P02", "P07"]
FORCE = []                 # stages to recompute even if their fingerprint is unchanged
OVERRIDES = {
    # "gpu": {"batch_size": 32},
    # "audio": {"enable_pitch": True},
    # "runtime": {"decode_threads": 6, "prefetch_batches": 2},
}

In [ ]:
from wildspeech.notebook_helpers import setup, show_stage

ctx, pipe = setup(PROFILE, OVERRIDES, root=ROOT)
print("device:", ctx.device, "| data:", ctx.paths.data_dir)
display(pipe.run(STAGES, force=FORCE, dry_run=True))   # plan only

In [ ]:
# ---- execute (resumable: interrupted stages continue from their Parquet checkpoints)
results = pipe.run(STAGES, force=FORCE)
display(results)

In [ ]:
display(pipe.status_table())

## Inspect results
Each stage writes `outputs/<Pxx>_<name>/tables/*.csv` and `figures/*.png` (every figure has a matching `fig_*.csv`).

In [ ]:
for sid in ["P00", "P01", "P10", "P13"]:
    show_stage(ctx, sid, max_rows=10, max_figures=4)

In [ ]:
display(ctx.registries.hypotheses.frame())
display(ctx.registries.experiments.frame().sort_values("r2_mean", ascending=False).head(20))

## Feature-family caches
Each acoustic family is cached independently in `cache/features/<family>/<split>/`. Changing one family's parameters
invalidates only that family. Clear one manually with `python -m wildspeech clear-cache --family pitch`.

In [ ]:
from wildspeech.acoustic.store import FeatureStore
display(FeatureStore(ctx.paths.cache / "features").status())

## Final submission (run only after reviewing P13)
Set `PROFILE = "SUBMISSION"` above and re-run the setup + execute cells, or run the cell below.

In [ ]:
RUN_SUBMISSION = False
if RUN_SUBMISSION:
    sctx, spipe = setup("SUBMISSION", root=ROOT)
    display(spipe.run(["P14"]))
    show_stage(sctx, "P14")